# NusaSynth - Metrik Single-LLM (baseline ablation)

Notebook TERPISAH dari `bab_iv_metrics.ipynb`. Fungsi metrik disalin **verbatim** demi komparabilitas; yang beda hanya loader (`syn_dfs` -> output Single-LLM). Bagian pipeline outcome & conflict TIDAK ADA (Single-LLM tanpa mesin/validator). Klaim = tingkat sistem.

## Setup

In [ ]:
import json, sys
from pathlib import Path

import numpy as np
import pandas as pd
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

pd.set_option('display.max_colwidth', 150)
pd.set_option('display.width', 200)

ROOT = Path('y:/Michh/Python/Projects/MAGenerator')
sys.path.insert(0, str(ROOT))
from NusaSynth.config import single_llm_synthetic_dir, synthetic_dir

LANGS = ['ace', 'ban', 'bjn', 'jav', 'mad', 'min', 'sun']
SENTENCES_PER_SEED = 5
BATCH_SIZE = 5

seed_dfs, syn_dfs = {}, {}
for lang in LANGS:
    seed_p = ROOT / f'data/nusax_senti/{lang}/train.csv'
    syn_p = single_llm_synthetic_dir(lang) / 'synthetic.csv'
    if not (seed_p.exists() and syn_p.exists()):
        print(f'  [{lang}] SKIP: file tidak lengkap ({syn_p})')
        continue
    seed_dfs[lang] = pd.read_csv(seed_p)
    syn_dfs[lang] = pd.read_csv(syn_p)
    print(f'  [{lang}] seed={len(seed_dfs[lang])} syn={len(syn_dfs[lang])}')
print('Sumber synthetic = SINGLE-LLM ->', single_llm_synthetic_dir(LANGS[0]).parent)

## B. Keberagaman & Kebaruan - Self-BLEU (pers. 3.11) & BLEU vs Seed (pers. 3.12)
> Fungsi identik `bab_iv_metrics.ipynb` (disalin verbatim).

In [ ]:
# B2 — Self-BLEU (pers. 3.11) & BLEU vs Seed (pers. 3.12)
# BLEU = rata-rata geometrik presisi n-gram termodifikasi, bobot seragam 1/N (Papineni et al. 2002); 4 orde kumulatif 2-5, BLEU-4 acuan.
# Implementasi ikut Texygen SelfBleu.py: bobot seragam + method1 smoothing (eps=0.1).
#
# BEST PRACTICE: Self-BLEU sensitif ukuran |D| (makin banyak referensi -> makin tinggi). Data yg sama:
#   |D|~900 -> ~0.22 ; |D|~190 -> ~0.11. Jadi Self-BLEU sintetis dihitung pada |D| DISETARAKAN ke seed,
#   lalu dibandingkan ke Self-BLEU seed (data manusia) sbg baseline. Sampel acak deterministik seed=42.
smoothie = SmoothingFunction().method1
ORDERS = (2, 3, 4, 5)
WEIGHTS = [tuple([1.0 / n] * n) for n in ORDERS]  # kumulatif orde 1..n; list-of-weights -> satu call semua orde
SAMPLE_SEED = 42


def tokenize(text):
    return text.lower().split()


def _bleu_orders(refs_tokens, hyp_tokens):
    # satu call sentence_bleu -> semua orde; BLEU-4 identik bit-per-bit dgn (0.25,)*4 tunggal
    return dict(zip(ORDERS, sentence_bleu(refs_tokens, hyp_tokens, WEIGHTS, smoothing_function=smoothie)))


def self_bleu(sentences):
    # pers. 3.11 pada himpunan D: tiap kalimat vs semua lainnya (label sama), dirata-rata
    if len(sentences) < 2:
        return {n: 0.0 for n in ORDERS}
    T = [tokenize(s) for s in sentences]
    acc = {n: [] for n in ORDERS}
    for i in range(len(T)):
        d = _bleu_orders([T[j] for j in range(len(T)) if j != i], T[i])
        for n in ORDERS:
            acc[n].append(d[n])
    return {n: float(np.mean(acc[n])) for n in ORDERS}


def bleu_vs_ref(hyps, refs):
    # pers. 3.12: tiap kalimat sintetis vs seluruh korpus seed label sama, dirata-rata
    if not hyps or not refs:
        return {n: 0.0 for n in ORDERS}
    R = [tokenize(r) for r in refs]
    acc = {n: [] for n in ORDERS}
    for h in hyps:
        d = _bleu_orders(R, tokenize(h))
        for n in ORDERS:
            acc[n].append(d[n])
    return {n: float(np.mean(acc[n])) for n in ORDERS}


def match_sample(items, n):
    # sampel acak deterministik ukuran n (menyetarakan |D| sintetis ke seed)
    if len(items) <= n:
        return list(items)
    rng = np.random.default_rng(SAMPLE_SEED)
    return [items[i] for i in rng.choice(len(items), n, replace=False)]

In [ ]:
REF_ORDER = 4  # orde acuan (konfigurasi standar N=4, Papineni et al. 2002)
METRICS = [('self_syn', 'Self-BLEU sintetis'), ('self_seed', 'Self-BLEU seed'), ('bs', 'BLEU vs Seed')]

rows_head, rows_multi = [], []
for lang in LANGS:
    if lang not in syn_dfs:
        continue
    seed_df, syn_df = seed_dfs[lang], syn_dfs[lang]
    pl = {}
    for label in ['negative', 'neutral', 'positive']:
        seed_t = seed_df[seed_df['label'] == label]['text'].dropna().tolist()
        syn_all = syn_df[syn_df['label'] == label]['text'].dropna().tolist()
        if len(syn_all) < 2 or len(seed_t) < 2:
            continue
        syn_m = match_sample(syn_all, len(seed_t))       # Self-BLEU: |D| sintetis = |D| seed (adil, pers. 3.11)
        pl[label] = {
            'self_syn':  self_bleu(syn_m),               # keberagaman sintetis (matched ke seed)
            'self_seed': self_bleu(seed_t),              # keberagaman seed (baseline manusia)
            'bs':        bleu_vs_ref(syn_all, seed_t),   # BLEU-vs-Seed: SELURUH sintetis vs seed (pers. 3.12)
        }
    if not pl:
        continue
    macro = lambda k, n: float(np.mean([pl[l][k][n] for l in pl]))

    # Headline BLEU-4 (orde acuan) — ketiga metrik berdampingan
    rows_head.append({
        'lang': lang,
        'selfBLEU_syn':  round(macro('self_syn',  REF_ORDER), 4),
        'selfBLEU_seed': round(macro('self_seed', REF_ORDER), 4),
        'bleu_vs_seed':  round(macro('bs',        REF_ORDER), 4),
    })
    # Multi-orde BLEU-2/3/4/5 — ketiga metrik (selaras Zhu 2018, Tabel 2 & 3)
    for key, name in METRICS:
        rows_multi.append({'lang': lang, 'metrik': name,
                           **{f'BLEU-{n}': round(macro(key, n), 4) for n in ORDERS}})

print(f'HEADLINE BLEU-{REF_ORDER} (orde acuan) | Self-BLEU: |D| disamakan ke seed (~190/label, seed=42) | BLEU-vs-Seed: seluruh sintetis (pers. 3.12)')
display(pd.DataFrame(rows_head))
print('\nMULTI-ORDE BLEU-2/3/4/5 (selaras Zhu 2018) — orde tinggi lebih peka pengulangan struktur panjang')
display(pd.DataFrame(rows_multi))

## C. Kesesuaian Distribusi Panjang - R_sigma (3.13) & Delta_mu (3.14)
> Single-LLM tanpa PSLM -> R_sigma diperkirakan jauh dari 1 (verdict 'CHECK' wajar, itu kontrafaktualnya).

In [ ]:
rows = []
for lang in LANGS:
    if lang not in syn_dfs:
        continue
    seed_df, syn_df = seed_dfs[lang].copy(), syn_dfs[lang].copy()
    seed_df['wc'] = seed_df['text'].str.split().str.len()
    syn_df['wc']  = syn_df['text'].str.split().str.len()
    for label in ['negative', 'neutral', 'positive']:
        s_seed = seed_df[seed_df['label'] == label]['wc']
        s_syn  = syn_df[syn_df['label'] == label]['wc']
        if len(s_seed) > 1 and len(s_syn) > 1:
            ratio = round(s_syn.std() / s_seed.std(), 2)
            mean_diff = round(s_syn.mean() - s_seed.mean(), 1)
            verdict = 'OK' if 0.80 <= ratio <= 1.20 and abs(mean_diff) <= 5 else 'CHECK'
            rows.append({
                'lang': lang,
                'label': label,
                'seed_n': len(s_seed),
                'syn_n': len(s_syn),
                'seed_mean': round(s_seed.mean(), 1),
                'syn_mean': round(s_syn.mean(), 1),
                'mean_diff': mean_diff,
                'seed_std': round(s_seed.std(), 1),
                'syn_std': round(s_syn.std(), 1),
                'std_ratio': ratio,
                'verdict': verdict,
            })

display(pd.DataFrame(rows))

In [ ]:
def len_stats(df):
    df = df.copy()
    df['wc'] = df['text'].str.split().str.len()
    return {
        'n': len(df),
        'mean': round(df['wc'].mean(), 1),
        'std': round(df['wc'].std(), 1),
        'min': int(df['wc'].min()),
        'max': int(df['wc'].max()),
        'median': int(df['wc'].median()),
        'p95': int(df['wc'].quantile(0.95)),
    }

rows = []
for lang in LANGS:
    if lang not in syn_dfs:
        continue
    rows.append({'lang': lang, 'split': 'seed', **len_stats(seed_dfs[lang])})
    rows.append({'lang': lang, 'split': 'syn',  **len_stats(syn_dfs[lang])})
    if lang in valid_dfs:
        rows.append({'lang': lang, 'split': 'valid', **len_stats(valid_dfs[lang])})
    if lang in test_dfs:
        rows.append({'lang': lang, 'split': 'test',  **len_stats(test_dfs[lang])})

display(pd.DataFrame(rows))

## D. Audit Classifier (GlotLID purity, NusaBERT label-drift, duplication)

In [ ]:
run_dir = single_llm_synthetic_dir(LANGS[0]).parent
audit_path = run_dir / 'audit_summary.json'
if audit_path.exists():
    audit = pd.DataFrame(json.loads(audit_path.read_text(encoding='utf-8')))
    print('Audit classifier Single-LLM (dari scripts/single_llm_audit.py):')
    display(audit)
    print('\nmean 7 bahasa: dup={:.2f}%  lang_confusion={:.2f}%  label_drift={:.2f}%'.format(
        audit['dup_rate'].mean(), audit['lang_confusion'].mean(), audit['label_drift'].mean()))
else:
    print(f'Belum ada {audit_path} -> jalankan dulu: uv run python scripts/single_llm_audit.py')

## E. Perbandingan vs NusaSynth Penuh

In [ ]:
def mu_selfbleu4(get_dir):
    vals = []
    for lang in LANGS:
        seed_df = seed_dfs.get(lang)
        if seed_df is None:
            continue
        p = get_dir(lang) / 'synthetic.csv'
        if not p.exists():
            continue
        syn_df = pd.read_csv(p)
        per = []
        for label in ['negative', 'neutral', 'positive']:
            seed_t = seed_df[seed_df['label'] == label]['text'].dropna().tolist()
            syn_all = syn_df[syn_df['label'] == label]['text'].dropna().tolist()
            if len(syn_all) < 2 or len(seed_t) < 2:
                continue
            per.append(self_bleu(match_sample(syn_all, len(seed_t)))[4])
        if per:
            vals.append(float(np.mean(per)))
    return round(float(np.mean(vals)), 4) if vals else None

print('Self-BLEU-4 mu (7 bahasa) - lebih RENDAH = lebih beragam:')
print('  Single-LLM      :', mu_selfbleu4(single_llm_synthetic_dir))
print('  NusaSynth penuh :', mu_selfbleu4(synthetic_dir))